# Module 07: LLD State Machines Scheduling Elevator Parking — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/elevator_system.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import elevator_system

classes = [n for n, o in inspect.getmembers(elevator_system, inspect.isclass)
           if o.__module__ == 'elevator_system']
functions = [n for n, o in inspect.getmembers(elevator_system, inspect.isfunction)
             if o.__module__ == 'elevator_system']

print('module   : elevator_system')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(elevator_system, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Single car look algorithm upward continuation

This is the module's own `test_single_car_look_algorithm_upward_continuation` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from elevator_system import (
    Direction,
    DoorState,
    ElevatorCar,
    ElevatorController,
)

car = ElevatorCar(car_id=1, min_floor=1, max_floor=10)
assert car.current_floor == 1
assert car.direction == Direction.IDLE

# Add stops at floor 3 and floor 5
car.add_destination(3)
car.add_destination(5)
assert car.direction == Direction.UP

# Step 1: moves to floor 2
f1 = car.step()
assert f1 == 2
assert car.door_state == DoorState.CLOSED

# Step 2: arrives at floor 3 -> opens doors!
f2 = car.step()
assert f2 == 3
assert car.door_state == DoorState.OPEN
assert 3 not in car.up_stops

# Step 3: closes doors, moves to floor 4
f3 = car.step()
assert f3 == 4
assert car.door_state == DoorState.CLOSED

# Step 4: arrives at floor 5 -> opens doors!
f4 = car.step()
assert f4 == 5
assert car.door_state == DoorState.OPEN
assert not car.has_pending_stops

print('PASSED: test_single_car_look_algorithm_upward_continuation')

## 3. 🔮 Prediction — commit before you run

An elevator is at floor 5 going up. Requests arrive for floor 2 and floor 8. Predict the service order under SCAN scheduling, then under naive FIFO.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_elevator_reverses_direction_when_topmost_request_served`, which tests exactly this property.


In [ ]:
car = ElevatorCar(car_id=1, min_floor=1, max_floor=10)
# On floor 1, request floor 4 (UP) and floor 2 (DOWN)
car.add_destination(4)
car.down_stops.add(2)  # Simulates someone requesting downward from floor 2

# Advance until floor 4 is reached
while car.current_floor < 4:
    car.step()

assert car.current_floor == 4
# With no higher UP stops, it should switch direction to DOWN to service floor 2
car.step()
assert car.direction == Direction.DOWN

print('PASSED: test_elevator_reverses_direction_when_topmost_request_served')

## 4. Measure it: Multi car controller assigns nearest idle car

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_multi_car_controller_assigns_nearest_idle_car` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

controller = ElevatorController(num_cars=3, min_floor=1, max_floor=20)
# Position cars at floors 1, 10, 18
controller.cars[0].current_floor = 1
controller.cars[1].current_floor = 10
controller.cars[2].current_floor = 18

# Hall call at floor 11 going UP should be assigned to Car 2 (current floor 10)
assigned = controller.request_elevator(floor=11, direction=Direction.UP)
assert assigned.car_id == 2
assert 11 in assigned.up_stops

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_multi_car_controller_assigns_nearest_idle_car')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(elevator_system) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. State machines make illegal transitions impossible rather than merely unlikely.
2. SCAN beats FIFO on total travel because it exploits direction locality.
3. Model the states first; the scheduling policy then has somewhere to live.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
